# PubMed Oncology Qwen3.6 27B SFT Fine-Tuning with Unsloth (4-bit QLoRA)

**Base Model:** unsloth/Qwen3.6-27B (bf16 weights, quantized to bnb 4-bit on load)

**Dataset:** PubMed oncology datagen JSONL — multi-turn QA, continuation, treatment reasoning, beyond-evidence, and self-correction conversations across 11 cancer types

**Training Hardware:** NVIDIA DGX Spark (128GB unified memory)

**Chat Template:** `<|im_start|>role\ncontent<|im_end|>` (ChatML)

**Architecture:** This is Phase 1 (SFT). Teaches the model clinical oncology reasoning with thinking chains. Phase 2 (DPO) refines response quality using preference pairs.

## How to run
**Press "Run All" and walk away.** Every cell is self-contained and runs in order.

## 1. Setup — Configuration, Environment, GPU Check

Installs missing packages, verifies GPU, sets all paths and hyperparameters. Safe to re-run.

In [ ]:
import os, sys, subprocess, importlib, importlib.util

# DGX Spark unified memory: reclaim cached CUDA blocks before the pool grows unchecked.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "garbage_collection_threshold:0.5,max_split_size_mb:256"

# =========================== HUGGING FACE CACHE ===========================
# Use the default Hub cache: /root/.cache/huggingface/hub in the unsloth-notebook container,
# which is bind-mounted from /home/spark/.cache/huggingface/hub on the host. Do NOT override
# HF_HUB_CACHE: pointing it at the cache root (not hub/) made unsloth build a second
# models--* tree beside hub/ and re-download every base model.

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 0: NEUTER GIT                                                         ║
# ║                                                                              ║
# ║  gptqmodel's startup banner shells out to `git rev-parse` to stamp its      ║
# ║  version. /workspace/* is owned by the host user but git runs as root in    ║
# ║  this container → "dubious ownership" failure, which trips up downstream    ║
# ║  code that wasn't expecting a non-zero git exit. We don't use git for       ║
# ║  anything here, so prepend a fake `git` to PATH that always exits 0.        ║
# ║  Must run before any heavy imports (unsloth → transformers → gptqmodel).    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
import tempfile, stat
_fake_git_dir = tempfile.mkdtemp(prefix="nogit-")
_fake_git = os.path.join(_fake_git_dir, "git")
with open(_fake_git, "w") as _f:
    _f.write("#!/bin/sh\nexit 0\n")
os.chmod(_fake_git, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
os.environ["PATH"] = _fake_git_dir + os.pathsep + os.environ.get("PATH", "")

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 1: INSTALL MISSING PACKAGES (safe to re-run, never clobbers NGC torch)║
# ║                                                                              ║
# ║  ORDER MATTERS:                                                              ║
# ║    1. torch check (no side effects)                                          ║
# ║    2. pip install small packages                                             ║
# ║    3. fix causal_conv1d (NGC ships broken build w/o CUDA extension)          ║
# ║    4. import unsloth FIRST (BEFORE transformers — required for patching)     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def _pip(*args, env_extra=None):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        print(f"  PIP FAILED: {' '.join(args)}")
        print(result.stderr[-500:] if result.stderr else result.stdout[-500:])
        return False
    return True

def _check_import(module_name):
    try:
        return importlib.import_module(module_name)
    except (ImportError, ModuleNotFoundError):
        return None

print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)

# ── 1a. Verify NGC CUDA PyTorch is intact ──
import torch
if not torch.cuda.is_available():
    print("FATAL: torch.cuda.is_available() = False")
    print(f"  torch version: {torch.__version__}")
    if "+cpu" in torch.__version__ or "cpu" in torch.__version__:
        print("  NGC CUDA PyTorch was clobbered by pip. Recreate the container.")
    else:
        print("  GPU not passed through. Check Portainer: runtime=nvidia, NVIDIA_VISIBLE_DEVICES=all")
    raise RuntimeError("No GPU. Cannot continue. See messages above.")

_MEMORY_FRACTION = 0.55
try:
    torch.cuda.set_per_process_memory_fraction(_MEMORY_FRACTION, 0)
    print(f"  CUDA allocator cap: {_MEMORY_FRACTION:.0%} of visible memory")
except RuntimeError as error:
    print(f"  CUDA allocator cap was not applied: {error}")

print(f"  torch {torch.__version__} — CUDA {torch.version.cuda} — GPU: {torch.cuda.get_device_name(0)}")

# ── 1b. Small utility packages ──
for module, install_args in {
    "psutil":      ["install", "-q", "psutil"],
    "matplotlib":  ["install", "-q", "matplotlib"],
    "ipywidgets":  ["install", "-q", "ipywidgets"],
    "torchvision": ["install", "-q", "--no-deps", "torchvision"],
    "PIL":         ["install", "-q", "pillow"],
}.items():
    if _check_import(module) is None:
        print(f"  Installing {install_args[-1]}...")
        _pip(*install_args)

# ── 1c. Fix causal_conv1d ──
_causal_ok = False
_build_env = {
    "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",
    "TORCH_CUDA_ARCH_LIST": "12.0;12.1",
}
try:
    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
    _causal_ok = True
    print("  causal_conv1d: OK (CUDA extension loaded)")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError, OSError):
    print("  causal_conv1d: CUDA extension missing — rebuilding from source (~3 min)...")
    _pip("uninstall", "-y", "causal-conv1d")
    _pip("cache", "remove", "causal_conv1d")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
    importlib.invalidate_caches()
    ok = _pip("install", "--no-build-isolation", "--no-deps", "--force-reinstall",
              "--no-binary", "causal-conv1d", "causal-conv1d", env_extra=_build_env)
    if ok:
        importlib.invalidate_caches()
        try:
            from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
            _causal_ok = True
            print("  causal_conv1d: rebuilt OK (CUDA extension working)")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
        except (ImportError, ModuleNotFoundError, OSError):
            print("  causal_conv1d: rebuild produced no CUDA ext — uninstalling for fallback")
            _pip("uninstall", "-y", "causal-conv1d")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
            importlib.invalidate_caches()
    else:
        print("  causal_conv1d: source build failed — uninstalling for fallback")
        _pip("uninstall", "-y", "causal-conv1d")
        importlib.invalidate_caches()

# ── 1d. Import unsloth FIRST, then transformers ──
for _k in list(sys.modules.keys()):
    if _k in ("transformers", "trl", "peft") or _k.startswith(("transformers.", "trl.", "peft.")):
        del sys.modules[_k]
importlib.invalidate_caches()

import unsloth
import transformers
print(f"  transformers {transformers.__version__}")

# ── 1e. Report final state ──
print()
for name, mod in [("unsloth", "unsloth"), ("transformers", "transformers"), ("trl", "trl"),
                   ("causal_conv1d", "causal_conv1d")]:
    m = _check_import(mod)
    v = getattr(m, "__version__", "installed") if m else "n/a"
    status = "OK" if m else "FALLBACK" if name == "causal_conv1d" else "MISSING"
    print(f"  {name:25s} {v:20s} [{status}]")

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 2: CONFIGURATION                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print()
print("=" * 60)
print("CONFIGURATION")
print("=" * 60)

# --- Paths (auto-detect Docker vs host) ---
if os.path.exists("/workspace/training/pubmed"):
    PROJECT_ROOT = "/workspace/training/pubmed"
    _env = "Docker (Unsloth container)"
elif os.path.exists("/workspace/pubmed"):
    PROJECT_ROOT = "/workspace/pubmed"
    _env = "Docker (legacy mount)"
else:
    PROJECT_ROOT = "/home/spark/projects/training/pubmed"
    _env = "Host (VS Code / venv)"

DATA_DIR    = f"{PROJECT_ROOT}/data"
OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

# =========================== MODEL CONFIGURATION ===========================
BASE_LLM        = "unsloth/Qwen3.6-27B"
MODEL_NAME_BASE = "pubmed_oncologist_v2_sft_qwen36_27b_unsloth_4bit"

# =========================== INPUT DATA ===========================
# Multi-turn SFT dataset (native messages or ShareGPT conversations format)
INPUT_DATA_FILE = f"{DATA_DIR}/training-data/pubmed_oncologist_v2/pubmed_oncologist_combined_sharegpt.jsonl"

# Cap on number of conversations used. 0 = use all loaded conversations.
# Applied AFTER loading the JSONL, BEFORE quality validation. Set e.g. 500
# for a smoke-test run, 3000 for a quick LoRA, leave 0 for full training.
SFT_MAX_EXAMPLES = 0

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR     = f"{OUTPUT_ROOT}/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"
LORA_OUTPUT_DIR     = f"{OUTPUT_BASE_DIR}/lora_adapters"

# =========================== TRAINING HYPERPARAMETERS ===========================
MAX_SEQ_LENGTH  = 16384
BATCH_SIZE      = 1
GRAD_ACCUM      = 8        

LEARNING_RATE   = 2e-4
TARGET_EPOCHS   = 1

# =========================== LoRA CONFIGURATION ===========================
LORA_R              = 32
LORA_ALPHA          = 32
LORA_DROPOUT        = 0
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# =========================== INFERENCE TEST ===========================
TEST_PROMPT = "A 58-year-old woman with BRCA1-mutated high-grade serous ovarian cancer has progressed after platinum-based chemotherapy and a PARP inhibitor. What are the next treatment options?"

# --- Print summary ---
print(f"  Environment:  {_env}")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  Base model:   {BASE_LLM}")
print(f"  Model name:   {MODEL_NAME_BASE}")
print(f"  Input data:   {INPUT_DATA_FILE}")
print(f"  SFT cap:      {SFT_MAX_EXAMPLES or 'ALL'}")
print(f"  LoRA output:  {LORA_OUTPUT_DIR}")
print(f"  LoRA:         r={LORA_R}, alpha={LORA_ALPHA}, targets={len(LORA_TARGET_MODULES)} modules")
print(f"  Training:     batch={BATCH_SIZE} x grad_accum={GRAD_ACCUM} = effective {BATCH_SIZE*GRAD_ACCUM}")
print(f"  Precision:    4-bit QLoRA (bnb quantized on load)")

# --- Verify paths ---
for path, label in [(INPUT_DATA_FILE, "Training data"), (PROJECT_ROOT, "Project root")]:
    exists = os.path.exists(path)
    print(f"  {'OK' if exists else 'MISSING':7s} {label}: {path}")
    if not exists:
        raise FileNotFoundError(f"{label} not found: {path}")

print()
print("Setup complete. All cells below can run sequentially.")


ENVIRONMENT SETUP
  CUDA allocator cap: 55% of visible memory
  torch 2.10.0a0+b558c986e8.nv25.11 — CUDA 13.0 — GPU: NVIDIA GB10
  causal_conv1d: OK (CUDA extension loaded)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
  transformers 5.15.0.dev0

  unsloth                   2026.7.5             [OK]
  transformers              5.15.0.dev0          [OK]
  trl                       0.24.0               [OK]
  causal_conv1d             0.0.local            [OK]

CONFIGURATION
  Environment:  Docker (Unsloth container)
  PROJECT_ROOT: /workspace/training/pubmed
  Base model:   unsloth/Qwen3.6-27B
  Model name:   pubmed_oncologist_v2_sft_qwen36_27b_unsloth_4bit
  Input data:   /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2/pubmed_oncologist_combined_sharegpt.jsonl
  SFT cap:      ALL
  LoRA output:  /workspace/training/pubmed/output/pubmed_oncologist_v2_sft_qwen36_27b_unsloth_4bit

## 2. Load Dataset

Load the combined multi-turn ShareGPT JSONL from datagen.

- 33,349 conversations across 11 cancer types
- Data types: QA (with thinking), continuation, treatment reasoning, beyond-evidence, self-correction
- Standard ShareGPT format: `[system, human, gpt, human, gpt, ...]`
- System prompts are extracted from the JSONL at load time (stays in sync with datagen)

In [2]:
import json, os, random
from collections import Counter, defaultdict

print("SCANNING SHAREGPT SFT DATA")
print(f"  File: {INPUT_DATA_FILE}")

cancer_type_counts = Counter()
source_counts = Counter()
line_indices_by_cancer = defaultdict(list)
bad_examples = []
empty_final_responses = []
system_prompts_by_cancer = {}


def _row_messages(row):
    if "messages" in row:
        return row["messages"]
    if "conversations" in row:
        role_map = {"system": "system", "human": "user", "gpt": "assistant"}
        return [
            {"role": role_map[turn["from"]], "content": turn["value"]}
            for turn in row["conversations"]
        ]
    raise ValueError("Row missing both 'messages' and 'conversations'")


with open(INPUT_DATA_FILE) as handle:
    for line_index, line in enumerate(handle):
        row = json.loads(line)
        cancer = row.get("cancer_type", "unknown")
        source = row.get("source", "unknown")
        cancer_type_counts[cancer] += 1
        source_counts[source] += 1

        try:
            messages = _row_messages(row)
        except (KeyError, ValueError) as error:
            bad_examples.append((line_index, str(error)))
            continue

        if len(messages) < 3:
            bad_examples.append((line_index, f"Expected >=3 messages, got {len(messages)}"))
            continue
        if messages[0].get("role") != "system" or messages[1].get("role") != "user":
            bad_examples.append((line_index, "Expected first two roles to be system,user"))
            continue
        if messages[-1].get("role") != "assistant" or not str(messages[-1].get("content", "")).strip():
            empty_final_responses.append(line_index)
            continue

        line_indices_by_cancer[cancer].append(line_index)
        system_prompts_by_cancer.setdefault(cancer, messages[0].get("content", ""))

eligible_count = sum(len(indices) for indices in line_indices_by_cancer.values())
if SFT_MAX_EXAMPLES < 0:
    raise ValueError(f"SFT_MAX_EXAMPLES must be 0 or positive, got {SFT_MAX_EXAMPLES}")

selected_line_indices = None
if SFT_MAX_EXAMPLES and eligible_count > SFT_MAX_EXAMPLES:
    rng = random.Random(42)
    cancers_sorted = sorted(line_indices_by_cancer, key=lambda cancer: -len(line_indices_by_cancer[cancer]))
    base = SFT_MAX_EXAMPLES // len(cancers_sorted)
    extras = SFT_MAX_EXAMPLES - base * len(cancers_sorted)
    selected_line_indices = set()
    for index, cancer in enumerate(cancers_sorted):
        pool = list(line_indices_by_cancer[cancer])
        rng.shuffle(pool)
        selected_line_indices.update(pool[:min(len(pool), base + (index < extras))])
    selected_count = len(selected_line_indices)
else:
    selected_count = eligible_count

print(f"  Total rows:       {sum(cancer_type_counts.values()):,}")
print(f"  Eligible rows:    {eligible_count:,}")
print(f"  Selected rows:    {selected_count:,}")
print(f"  Bad structure:    {len(bad_examples):,}")
print(f"  Empty final turn: {len(empty_final_responses):,}")


SCANNING SHAREGPT SFT DATA
  File: /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2/pubmed_oncologist_combined_sharegpt.jsonl
  Total rows:       40,020
  Eligible rows:    40,020
  Selected rows:    40,020
  Bad structure:    0
  Empty final turn: 0


## 3. Validate & Summarize Dataset

Verify data quality: turn structure, non-empty responses, thinking tag presence.

In [3]:
print("DATA QUALITY SUMMARY")
print(f"  Selected conversations: {selected_count:,}")
print(f"  Cancer types: {len(cancer_type_counts)}")
print(f"  Sources: {len(source_counts)}")

print("\nPer-cancer breakdown (source file):")
for cancer, count in cancer_type_counts.most_common():
    print(f"  {cancer:30s} {count:>5d} conversations")

print("\nPer-source breakdown (source file):")
for source, count in source_counts.most_common():
    print(f"  {source:30s} {count:>5d} rows")

if bad_examples:
    print("\nBad examples (first 5):")
    for line_index, reason in bad_examples[:5]:
        print(f"  Line {line_index + 1}: {reason}")

print("\nDataset scan complete. Tokenization will stream from disk in Section 5.")


DATA QUALITY SUMMARY
  Selected conversations: 40,020
  Cancer types: 11
  Sources: 1

Per-cancer breakdown (source file):
  multi                           4802 conversations
  pubmed_brain_tumour             4098 conversations
  pubmed_bone_cancer              3877 conversations
  pubmed_colon_cancer             3783 conversations
  pubmed_ovarian_cancer           3728 conversations
  pubmed_breast_cancer            3689 conversations
  pubmed_lung_cancer              3340 conversations
  pubmed_skin_cancer              3264 conversations
  pubmed_kidney_cancer            3249 conversations
  pubmed_gastric_cancer           3145 conversations
  pubmed_prostate_cancer          3045 conversations

Per-source breakdown (source file):
  unknown                        40020 rows

Dataset scan complete. Tokenization will stream from disk in Section 5.


## 4. Load Model & Tokenizer (4-bit)

In [4]:
import re
import torch

# Disable Unsloth's broken Qwen3.6 compile/flex-attention path before importing it.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
from unsloth import FastLanguageModel

# ── Work around broken optional gptqmodel dependency ─────────────────────────
# The installed gptqmodel cannot import with this Transformers version, but
# PEFT imports its AWQ class merely because the package is installed.
import types

class _AwqGEMMQuantLinearStub:
    pass

_gptqmodel = types.ModuleType("gptqmodel")
_gptqmodel.__path__ = []
_gptqmodel_nn_modules = types.ModuleType("gptqmodel.nn_modules")
_gptqmodel_nn_modules.__path__ = []
_gptqmodel_qlinear = types.ModuleType("gptqmodel.nn_modules.qlinear")
_gptqmodel_qlinear.__path__ = []
_gptqmodel_gemm_awq = types.ModuleType("gptqmodel.nn_modules.qlinear.gemm_awq")
_gptqmodel_gemm_awq.AwqGEMMQuantLinear = _AwqGEMMQuantLinearStub

sys.modules.update({
    "gptqmodel": _gptqmodel,
    "gptqmodel.nn_modules": _gptqmodel_nn_modules,
    "gptqmodel.nn_modules.qlinear": _gptqmodel_qlinear,
    "gptqmodel.nn_modules.qlinear.gemm_awq": _gptqmodel_gemm_awq,
})
# ───────────────────────────────────────────────────────────────────────────────

# Quantize the bf16 base to bnb 4-bit on load for QLoRA.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Qwen3.6-27B is a VLM — Unsloth returns a Qwen3VLProcessor (multimodal wrapper),
# not a raw tokenizer. For text-only SFT, extract the inner tokenizer so that
# chat template / SFTTrainer get a real PreTrainedTokenizer.
if hasattr(tokenizer, "tokenizer"):
    processor = tokenizer
    tokenizer = processor.tokenizer
    print("  (Extracted tokenizer from processor — text-only SFT mode)")

# Fix pad token — set pad = eos for causal LM training.
if tokenizer.pad_token is None or tokenizer.pad_token_id != tokenizer.eos_token_id:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"  Set pad_token = eos_token ({tokenizer.eos_token!r}, id={tokenizer.eos_token_id})")

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
if hasattr(model, "generation_config"):
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id

print(f"Model loaded: {BASE_LLM}")
print(f"  Precision: 4-bit QLoRA (bnb quantized on load)")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Vocab size: {len(tokenizer)}")
print(f"  GPU allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")


==((====))==  Unsloth 2026.7.5: Fast Qwen3_5 patching. Transformers: 5.15.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

  (Extracted tokenizer from processor — text-only SFT mode)
  Set pad_token = eos_token ('<|im_end|>', id=248046)
Model loaded: unsloth/Qwen3.6-27B
  Precision: 4-bit QLoRA (bnb quantized on load)
  Max sequence length: 32768
  Vocab size: 248077
  GPU allocated: 18.0 GB


## 5. Format Dataset for Chat Template

Map ShareGPT roles to chat template roles, apply Qwen3's native chat template (ChatML), then manual sequence packing for 100% token utilization.

**Note:** `enable_thinking=False` is used in `apply_chat_template` because the training data already contains `<think>` blocks in the GPT response text. Without this flag, the template would inject a second `<think>` tag.


In [5]:
import bisect
import gc
import hashlib
import json
import os
import shutil
from pathlib import Path

import torch
from torch.utils.data import Dataset

TOKEN_CACHE_DIR = Path(OUTPUT_BASE_DIR) / "packed_token_cache"
TOKEN_SHARD_DIR = TOKEN_CACHE_DIR / "shards"
TOKEN_MANIFEST_PATH = TOKEN_CACHE_DIR / "manifest.json"
TOKEN_STATE_PATH = TOKEN_CACHE_DIR / "build_state.pt"
TOKEN_SHARD_CHUNKS = 32

cache_config = {
    "cache_version": 1,
    "input_file": str(Path(INPUT_DATA_FILE).resolve()),
    "input_size": Path(INPUT_DATA_FILE).stat().st_size,
    "input_mtime_ns": Path(INPUT_DATA_FILE).stat().st_mtime_ns,
    "selected_count": selected_count,
    "sft_max_examples": SFT_MAX_EXAMPLES,
    "max_seq_length": MAX_SEQ_LENGTH,
    "tokenizer_name": getattr(tokenizer, "name_or_path", type(tokenizer).__name__),
    "tokenizer_class": type(tokenizer).__name__,
    "tokenizer_vocab_size": len(tokenizer),
    "eos_token_id": tokenizer.eos_token_id,
    "chat_template_sha256": hashlib.sha256(str(tokenizer.chat_template).encode("utf-8")).hexdigest(),
    "enable_thinking": False,
    "packing": "continuous_eos_separated",
    "shard_chunks": TOKEN_SHARD_CHUNKS,
}
cache_fingerprint = hashlib.sha256(
    json.dumps(cache_config, sort_keys=True).encode("utf-8")
).hexdigest()


def _read_manifest():
    if not TOKEN_MANIFEST_PATH.exists():
        return None
    with open(TOKEN_MANIFEST_PATH) as handle:
        return json.load(handle)


def _atomic_json_dump(payload, destination):
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    with open(temporary, "w") as handle:
        json.dump(payload, handle, indent=2)
    os.replace(temporary, destination)


def _atomic_torch_save(payload, destination):
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, destination)


manifest = _read_manifest()
cache_complete = (
    manifest is not None
    and manifest.get("fingerprint") == cache_fingerprint
    and manifest.get("status") == "complete"
)

if not cache_complete:
    TOKEN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    TOKEN_SHARD_DIR.mkdir(parents=True, exist_ok=True)

    if manifest is None or manifest.get("fingerprint") != cache_fingerprint:
        shutil.rmtree(TOKEN_SHARD_DIR, ignore_errors=True)
        TOKEN_SHARD_DIR.mkdir(parents=True, exist_ok=True)
        if TOKEN_STATE_PATH.exists():
            TOKEN_STATE_PATH.unlink()
        manifest = {
            "status": "building",
            "fingerprint": cache_fingerprint,
            "config": cache_config,
            "next_line_index": 0,
            "selected_processed": 0,
            "total_tokens": 0,
            "shards": [],
        }
        _atomic_json_dump(manifest, TOKEN_MANIFEST_PATH)

    if TOKEN_STATE_PATH.exists():
        state = torch.load(TOKEN_STATE_PATH, map_location="cpu")
        token_buffer = state["token_buffer"].tolist()
    else:
        token_buffer = []

    pending_chunks = []
    next_line_index = manifest["next_line_index"]
    selected_processed = manifest["selected_processed"]
    total_tokens = manifest["total_tokens"]
    shard_records = list(manifest["shards"])

    def flush_pending_chunks():
        global pending_chunks, shard_records
        if not pending_chunks:
            return
        shard_index = len(shard_records)
        shard_path = TOKEN_SHARD_DIR / f"shard-{shard_index:06d}.pt"
        shard_tensor = torch.tensor(pending_chunks, dtype=torch.int32)
        _atomic_torch_save(shard_tensor, shard_path)
        shard_records.append({"file": shard_path.name, "chunks": len(pending_chunks)})
        pending_chunks = []

    with open(INPUT_DATA_FILE) as handle:
        for line_index, line in enumerate(handle):
            if line_index < next_line_index:
                continue
            if selected_line_indices is not None and line_index not in selected_line_indices:
                next_line_index = line_index + 1
                continue

            row = json.loads(line)
            try:
                messages = _row_messages(row)
            except (KeyError, ValueError):
                next_line_index = line_index + 1
                continue
            if (
                len(messages) < 3
                or messages[0].get("role") != "system"
                or messages[1].get("role") != "user"
                or messages[-1].get("role") != "assistant"
                or not str(messages[-1].get("content", "")).strip()
            ):
                next_line_index = line_index + 1
                continue

            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                enable_thinking=False,
            )
            token_ids = tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"]
            token_ids.append(tokenizer.eos_token_id)
            token_buffer.extend(token_ids)
            total_tokens += len(token_ids)
            selected_processed += 1
            next_line_index = line_index + 1

            while len(token_buffer) >= MAX_SEQ_LENGTH:
                pending_chunks.append(token_buffer[:MAX_SEQ_LENGTH])
                del token_buffer[:MAX_SEQ_LENGTH]

            if len(pending_chunks) >= TOKEN_SHARD_CHUNKS:
                flush_pending_chunks()
                _atomic_torch_save(
                    {"token_buffer": torch.tensor(token_buffer, dtype=torch.int32)},
                    TOKEN_STATE_PATH,
                )
                manifest.update({
                    "next_line_index": next_line_index,
                    "selected_processed": selected_processed,
                    "total_tokens": total_tokens,
                    "shards": shard_records,
                })
                _atomic_json_dump(manifest, TOKEN_MANIFEST_PATH)
                print(
                    f"  Cached {selected_processed:,}/{selected_count:,} conversations, "
                    f"{sum(record['chunks'] for record in shard_records):,} chunks",
                    flush=True,
                )

    flush_pending_chunks()
    wasted_tokens = len(token_buffer)
    if TOKEN_STATE_PATH.exists():
        TOKEN_STATE_PATH.unlink()
    manifest.update({
        "status": "complete",
        "next_line_index": next_line_index,
        "selected_processed": selected_processed,
        "total_tokens": total_tokens,
        "wasted_tokens": wasted_tokens,
        "shards": shard_records,
        "total_chunks": sum(record["chunks"] for record in shard_records),
    })
    _atomic_json_dump(manifest, TOKEN_MANIFEST_PATH)
else:
    print(f"Using existing packed-token cache: {TOKEN_CACHE_DIR}")

manifest = _read_manifest()
if manifest is None or manifest.get("status") != "complete":
    raise RuntimeError("Packed-token cache is incomplete")


class PackedTokenShardDataset(Dataset):
    def __init__(self, shard_dir, shard_records):
        self.shard_dir = Path(shard_dir)
        self.shard_records = shard_records
        self.cumulative_chunks = []
        total_chunks = 0
        for record in shard_records:
            total_chunks += record["chunks"]
            self.cumulative_chunks.append(total_chunks)
        self._loaded_shard_index = None
        self._loaded_shard = None

    def __len__(self):
        return self.cumulative_chunks[-1] if self.cumulative_chunks else 0

    def __getitem__(self, index):
        if index < 0:
            index += len(self)
        if index < 0 or index >= len(self):
            raise IndexError(index)

        shard_index = bisect.bisect_right(self.cumulative_chunks, index)
        shard_start = 0 if shard_index == 0 else self.cumulative_chunks[shard_index - 1]
        if self._loaded_shard_index != shard_index:
            shard_path = self.shard_dir / self.shard_records[shard_index]["file"]
            self._loaded_shard = torch.load(shard_path, map_location="cpu")
            self._loaded_shard_index = shard_index

        chunk = self._loaded_shard[index - shard_start]
        return {"input_ids": chunk.to(torch.long)}


def packed_token_collator(features):
    input_ids = torch.stack([feature["input_ids"] for feature in features])
    return {
        "input_ids": input_ids,
        "attention_mask": torch.ones_like(input_ids),
        "labels": input_ids.clone(),
    }


dataset = PackedTokenShardDataset(TOKEN_SHARD_DIR, manifest["shards"])
if len(dataset) == 0:
    raise RuntimeError("Packed-token cache contains no complete chunks")

gc.collect()
torch.cuda.empty_cache()
print(f"Packed-token cache ready: {len(dataset):,} chunks of {MAX_SEQ_LENGTH:,} tokens")
print(f"  Shards: {len(manifest['shards']):,} at {TOKEN_SHARD_DIR}")
print(f"  Conversations tokenized: {manifest['selected_processed']:,}")
print(f"  Total tokens: {manifest['total_tokens']:,}")
print(f"  Tail discarded: {manifest['wasted_tokens']:,} tokens")
print("  Runtime RAM: one token shard plus the active DataLoader batch")


Using existing packed-token cache: /workspace/training/pubmed/output/pubmed_oncologist_v2_sft_qwen36_27b_unsloth_4bit/packed_token_cache
Packed-token cache ready: 3,101 chunks of 32,768 tokens
  Shards: 97 at /workspace/training/pubmed/output/pubmed_oncologist_v2_sft_qwen36_27b_unsloth_4bit/packed_token_cache/shards
  Conversations tokenized: 40,020
  Total tokens: 101,636,576
  Tail discarded: 23,008 tokens
  Runtime RAM: one token shard plus the active DataLoader batch


## 6. Add LoRA Adapters

### Freezing the vision / audio towers — deliberately, not by accident

`unsloth/Qwen3.6-27B` is a multimodal checkpoint: alongside the language model it carries a vision tower and an MTP head. This notebook trains a **text-only** persona LoRA, so none of that should receive an adapter.

That does **not** happen for free. PEFT matches a `target_modules` list like `["q_proj", "k_proj", ...]` by **suffix, across the entire module tree** — it has no notion of "the language model". Whether a tower gets caught depends purely on whether it reuses those leaf names:

- **Gemma 4** towers use `self_attn.{q,k,v,o}_proj` and `mlp.{gate,up,down}_proj` — identical to the language model, so a bare list catches every one of them.
- **Qwen VL** towers use `attn.qkv` / `attn.proj` / `linear_fc1` — no collision, so they escape. By luck of naming, not by design.

Two adapters in this workspace shipped with 224 vision and 72 audio LoRA tensors attached this way. They were harmless in effect — `lora_B` is zero-initialised and text batches never invoke those towers, so every leaked tensor stayed exactly zero — but they were 36–42% dead weight, and **vLLM refuses to load an adapter carrying vision-layer LoRA**.

The `finetune_*` flags below make the scoping explicit, and the assertion after the call proves it held rather than assuming it.

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    # ---- TOWER FREEZE: deliberate, not incidental ----------------------------------
    # These four flags are load-bearing. Unsloth routes an explicit target_modules LIST
    # through its language-scoped regex ONLY when at least one of them is False; all four
    # default to True, so a bare list goes straight to PEFT, which matches by SUFFIX
    # across the WHOLE module tree. finetune_vision_layers=False is what confines the
    # match to the language model - it excludes the audio tower and MTP head too, since
    # the scoped regex requires a `language`/`text` path segment.
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    # Deliberately True, NOT "unsloth". VERIFIED in
    # unsloth_zoo/gradient_checkpointing.py: the "unsloth" path offloads saved
    # activations to PINNED host memory (torch.empty(..., device="cpu",
    # pin_memory=True)) and grows those buffers with `if new_size > x.numel():
    # x.resize_(new_size)` - up only. The sole shrink path is
    # unpatch_unsloth_smart_gradient_checkpointing(), which runs at teardown, never
    # mid-run. On a discrete GPU that trade is a win: VRAM is scarce, host RAM is not.
    # On GB10 host and device are the SAME 128 GB pool, so the copy frees nothing and
    # adds a second, page-locked, monotonically growing copy the OS cannot reclaim.
    use_gradient_checkpointing=True,
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"LoRA adapters added (r={LORA_R}, alpha={LORA_ALPHA})")
print(f"  Trainable: {trainable:,} / {total:,} params ({100*trainable/total:.2f}%)")
print(f"  Target modules: {LORA_TARGET_MODULES}")

# ============ VERIFY THE ADAPTER SCOPE AND THE TOWER FREEZE ============
# This is a TEXT-ONLY fine-tune on a multimodal base. Two properties must hold, and they
# fail in different ways, so both are checked:
#
#   1. WHERE ADAPTERS ATTACHED. PEFT matches a target_modules LIST by suffix across the
#      whole module tree - it has no concept of "the language model". Any tower reusing
#      the language model's leaf names (q_proj, gate_proj, ...) collects adapters too.
#   2. WHAT IS TRAINABLE. Under QLoRA base weights are frozen 4-bit and only LoRA A/B
#      carry gradients, so the towers are frozen by construction. Assert it anyway - an
#      assumption that is never checked is an assumption that eventually breaks.
#
# See docs/multimodal_and_hybrid_base_models.md.
from collections import Counter as _Counter


def _zone_of(param_name):
    """Return the non-language sub-model a parameter belongs to, or None."""
    n = param_name.replace("base_model.model.", "", 1)
    if n.startswith("mtp."):
        return "mtp head"
    if "vision_tower" in n or ".visual." in n or n.startswith("visual."):
        return "vision tower"
    if "audio_tower" in n or n.startswith("audio_tower."):
        return "audio tower"
    if "multi_modal_projector" in n or "embed_vision" in n or "embed_audio" in n:
        return "mm projector"
    return None


_adapted = sorted({
    n.split(".lora_A")[0].split(".lora_B")[0].replace("base_model.model.", "")
    for n, _ in model.named_parameters() if ".lora_A." in n or ".lora_B." in n
})
_stray = [n for n in _adapted if _zone_of(n)]
if _stray:
    raise RuntimeError(
        f"LoRA attached to {len(_stray)} module(s) outside the language model "
        f"(e.g. {_stray[:3]}). The finetune_* scoping flags did not take effect. "
        "Do not train - vLLM refuses adapters carrying vision-layer LoRA."
    )

_unfrozen, _zone_params = {}, _Counter()
for _n, _p in model.named_parameters():
    _z = _zone_of(_n)
    if _z is None:
        continue
    _zone_params[_z] += 1
    if _p.requires_grad:
        _unfrozen.setdefault(_z, []).append(_n)

print(f"  LoRA adapted {len(_adapted)} modules, all inside the language model.")
for _z, _c in sorted(_zone_params.items()):
    _bad = len(_unfrozen.get(_z, []))
    print(f"  {_z:<14} {_c:>5} params  "
          f"{'FROZEN' if not _bad else str(_bad) + ' TRAINABLE - BAD'}")
if _unfrozen:
    raise RuntimeError(
        f"{sum(len(v) for v in _unfrozen.values())} tower/MTP parameter(s) are trainable "
        f"(e.g. {[n for v in _unfrozen.values() for n in v][:3]}). This is a text-only "
        "fine-tune - they must stay frozen."
    )


Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
LoRA adapters added (r=32, alpha=32)
  Trainable: 159,383,552 / 15,135,298,800 params (1.05%)
  Target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


## 7. Trainer Setup

In [7]:
import gc

from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig


class DgxUnifiedMemoryCallback(TrainerCallback):
    def on_step_begin(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()
        gc.collect()
        torch.cuda.reset_peak_memory_stats()

    def on_step_end(self, args, state, control, **kwargs):
        gib = 1024 ** 3
        allocated = torch.cuda.memory_allocated() / gib
        reserved = torch.cuda.memory_reserved() / gib
        peak = torch.cuda.max_memory_allocated() / gib
        print(
            f"  CUDA memory after step {state.global_step}: "
            f"allocated={allocated:.1f} GiB, reserved={reserved:.1f} GiB, peak={peak:.1f} GiB",
            flush=True,
        )
        torch.cuda.empty_cache()
        gc.collect()


trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    data_collator=packed_token_collator,
    args=SFTConfig(
        max_length=MAX_SEQ_LENGTH,
        packing=False,
        dataset_kwargs={"skip_prepare_dataset": True},
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        gradient_checkpointing=True,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=3,
        report_to="none",
        dataset_num_proc=1,
        dataloader_pin_memory=False,
        dataloader_num_workers=0,
    ),
)
trainer.add_callback(DgxUnifiedMemoryCallback())

effective_batch_size = BATCH_SIZE * GRAD_ACCUM
num_steps = (len(dataset) * TARGET_EPOCHS + effective_batch_size - 1) // effective_batch_size
print(f"Trainer configured (PubMed Oncology Qwen3.6 27B 4-bit QLoRA — pre-packed)")
print(f"  Effective batch size: {BATCH_SIZE} x {GRAD_ACCUM} = {effective_batch_size}")
print(f"  Epochs: {TARGET_EPOCHS}  |  Steps: ~{num_steps}")
print(f"  LR: {LEARNING_RATE}")
print(f"  Packing: disk-cached token shards ({MAX_SEQ_LENGTH} tokens each, zero padding)")
print(f"  Precision: {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")
print(f"  Dataset: {len(dataset)} packed chunks")


# ===================== TRAIN ON RESPONSES ONLY =====================
# Mask prompt tokens to -100 so loss is computed ONLY on assistant turns.
# Without this the loss averages in the system prompt and the user turns - tokens the
# base model already predicts near-perfectly and that repeat across every row. They pin
# the average near their own ~0 loss and bury the signal from the response tokens.
# Persona datasets are the worst case: the system prompt is ~1,600 characters and is
# byte-identical across thousands of examples.
#
# Both marker args are left as None so Unsloth auto-detects them from this model's chat
# template and prints what it found - no hand-written marker strings to get wrong.
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(trainer)

# Verify the mask. If the markers do not match the template every label becomes -100,
# the run trains on nothing, and it fails silently for the entire duration.
import numpy as np

_probe = trainer.train_dataset[:8]["labels"]
_kept = sum(int((np.array(x) != -100).sum()) for x in _probe)
_total = sum(len(x) for x in _probe)
if _kept == 0:
    raise RuntimeError(
        "train_on_responses_only masked EVERY token - the instruction/response markers "
        "did not match the chat template. Do not start training."
    )
print(f"Response masking OK: {_kept:,}/{_total:,} label tokens kept "
      f"({100 * _kept / _total:.1f}%) across 8 sample sequences")


Trainer configured (PubMed Oncology Qwen3.6 27B 4-bit QLoRA — pre-packed)
  Effective batch size: 1 x 8 = 8
  Epochs: 1  |  Steps: ~388
  LR: 0.0002
  Packing: disk-cached token shards (32768 tokens each, zero padding)
  Precision: bf16
  Dataset: 3101 packed chunks


## 8. Train

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

last_ckpt = get_last_checkpoint(trainer.args.output_dir)
if last_ckpt is not None:
    print(f"Resuming from checkpoint: {last_ckpt}")
    result = trainer.train(resume_from_checkpoint=True)
else:
    print("No previous checkpoint found — starting fresh.")
    result = trainer.train()

print(f"\n✓ Training complete!")
print(f"  Final loss:     {result.training_loss:.4f}")
print(f"  Total steps:    {result.global_step}")
print(f"  Training time:  {result.metrics.get('train_runtime', 0) / 60:.1f} minutes")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


No previous checkpoint found — starting fresh.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,101 | Num Epochs = 1 | Total steps = 388
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 159,383,552 of 27,516,112,112 (0.58% trained)


## 9. Save LoRA Adapters

Save the trained LoRA adapters and system prompts. The DPO notebook (Phase 2) expects the LoRA at this path.

In [ ]:
import json
from pathlib import Path

Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

# Save system prompts alongside adapters for inference use
prompts_path = f"{LORA_OUTPUT_DIR}/oncologist_system_prompts.json"
with open(prompts_path, "w") as f:
    json.dump(system_prompts_by_cancer, f, indent=2)

print(f"\nLoRA adapters saved!")
print(f"  Adapters:       {LORA_OUTPUT_DIR}")
print(f"  System prompts: {prompts_path} ({len(system_prompts_by_cancer)} cancer types)")
print(f"  DPO notebook expects LoRA at: {{PROJECT_ROOT}}/output/pubmed_oncologist_v2_sft_qwen36_27b_unsloth_4bit/lora_adapters")

for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")


## 10. Test Inference

Smoke test with oncology questions across different cancer types. This is a thinking model — responses will include `<think>` reasoning blocks.

In [ ]:
import warnings, logging, json
from transformers import TextStreamer

# Suppress transformers deprecation warning bug (msg % FutureWarning formatting error)
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("transformers.modeling_attn_mask_utils").setLevel(logging.ERROR)

# Load model from saved adapters if not already in memory
if "model" not in dir() or model is None:
    from unsloth import FastLanguageModel
    print(f"Model not in memory — loading from saved adapters: {LORA_OUTPUT_DIR}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=LORA_OUTPUT_DIR,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    # Qwen3.6-27B VLM — extract inner tokenizer from processor wrapper
    if hasattr(tokenizer, "tokenizer"):
        tokenizer = tokenizer.tokenizer

if "system_prompts_by_cancer" not in dir() or not system_prompts_by_cancer:
    with open(f"{LORA_OUTPUT_DIR}/oncologist_system_prompts.json") as f:
        system_prompts_by_cancer = json.load(f)

FastLanguageModel.for_inference(model)

test_cancers = list(system_prompts_by_cancer.keys())[:3]

print(f"INFERENCE TEST — {len(test_cancers)} CANCER TYPES\n")

for cancer_type in test_cancers:
    system_prompt = system_prompts_by_cancer[cancer_type]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": TEST_PROMPT},
    ]

    # Thinking model: let Qwen3 default (enable_thinking=True) inject <think>
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    print(f"{'='*60}")
    print(f"  CANCER TYPE: {cancer_type.upper()}")
    print(f"  Q: {TEST_PROMPT}")
    print(f"  A: ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print()

del inputs, outputs


## 11. Verify Adapter (Cold Reload)

Loads adapters from disk in a fresh model to confirm portability. Also verifies the DPO notebook can load these adapters.

In [ ]:
import gc, torch, json
from pathlib import Path

del model, tokenizer, trainer, dataset
gc.collect()
torch.cuda.empty_cache()

print("Cleared training model from memory")
print(f"  Loading adapter from: {LORA_OUTPUT_DIR}")

from unsloth import FastLanguageModel

model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
# Qwen3.6-27B VLM — extract inner tokenizer from processor wrapper
if hasattr(tokenizer2, "tokenizer"):
    tokenizer2 = tokenizer2.tokenizer

FastLanguageModel.for_inference(model2)

with open(f"{LORA_OUTPUT_DIR}/oncologist_system_prompts.json") as f:
    reloaded_prompts = json.load(f)

test_cancer = list(reloaded_prompts.keys())[0]
test_system = reloaded_prompts[test_cancer]

messages = [
    {"role": "system", "content": test_system},
    {"role": "user", "content": TEST_PROMPT},
]

text = tokenizer2.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer2(text, return_tensors="pt").to(model2.device)

outputs = model2.generate(
    **inputs,
    max_new_tokens=2048,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    do_sample=True,
)

response = tokenizer2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(f"\nADAPTER RELOAD TEST (cancer type: {test_cancer}):")
print(f"  Q: {TEST_PROMPT}")
print(f"  A: {response[:500]}")
print(f"\nAdapter loads cleanly from disk.")
print(f"DPO notebook (Phase 2) can now load this LoRA from: {LORA_OUTPUT_DIR}")

print(f"\nAdapter contents:")
for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

del model2, tokenizer2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()
